# Version 2 — Give the Agent Hands: PostgreSQL Tool Calling

We connect a `run_query(sql)` tool to PostgreSQL. The agent can now write SQL, run it, read the result and answer from it.

## Workflow

```text
User Question
     |
     v
AI Data Analyst Agent
     |
     v
Generate read-only SQL
     |
     v
run_query(sql) ───────────► PostgreSQL
     |                          |
     +<── rows / SQL error ─────+
     |
     v
Inspect result; correct and retry if needed
     |
     v
Final Answer
```

**Tools:** `run_query(sql)` only.

The model does not connect to PostgreSQL. It asks for a tool call; Python runs the SQL and hands the result back.

## What changes from Version 1

| | Version 1 | Version 2 |
|---|---|---|
| Tools | none | `run_query(sql)` |
| Execute SQL | No | Yes, through the tool |
| See real results | No | Yes |
| Prompt | proposes SQL | must run SQL and answer from the result |
| Output | QUESTION / PROPOSED SQL / … | ANSWER / DENOMINATOR / SOURCE / CAVEAT |

A prompt is not a security boundary. The tool and the database role must enforce read-only access.

## 1. Install dependencies

In [ ]:
%pip install -q langchain langchain-openai python-dotenv "psycopg[binary]"

## 2. Configure the model and PostgreSQL

| Variable | Required | Used for |
|---|---|---|
| `OPENAI_API_KEY` | yes | The chat model |
| `PGHOST`, `PGDATABASE`, `PGUSER`, `PGPASSWORD` | yes | PostgreSQL connection (use a read-only role) |
| `PGPORT` | no (default `5432`) | |
| `PGSSLMODE` | no | e.g. `require` if your provider needs TLS |

Put these in a `.env` file next to the notebook. Never commit credentials.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Configuration — Version 2
#
#   .env ──► load_dotenv ──► check keys ──► model (the brain)
#                                      └──► PostgreSQL settings (used by the tool)
#
# Does NOT open a database connection here. The tool opens one per call.
# ─────────────────────────────────────────────────────────────
import os
from dotenv import load_dotenv
import psycopg
from psycopg.rows import dict_row

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("Please set OPENAI_API_KEY in your environment or .env file.")

required_pg_vars = ["PGHOST", "PGDATABASE", "PGUSER", "PGPASSWORD"]
missing_pg_vars = [name for name in required_pg_vars if not os.environ.get(name)]
if missing_pg_vars:
    raise ValueError("Missing PostgreSQL environment variables: " + ", ".join(missing_pg_vars))

PGPORT = int(os.environ.get("PGPORT", "5432"))
PGSSLMODE = os.environ.get("PGSSLMODE")  # Optional, e.g. "require".

MODEL = "gpt-6-astra"
model = ChatOpenAI(model=MODEL)

## 3. The `run_query(sql)` tool — the agent's hands

- Accepts one query starting with `SELECT` or `WITH`.
- Runs it on a **read-only** connection with a 15-second timeout.
- Returns at most 50 rows, and says so when more rows exist.
- Returns SQL errors as text so the model can fix its query.

**Security note:** a first-word check is not a SQL validator. The read-only database role is the real boundary. Version 4 adds a proper SQL policy.

In [ ]:
# ─────────────────────────────────────────────────────────────
# PostgreSQL helpers (shared by the SQL tools)
#
#   pg_connect()   ──► new connection, read-only by default, with a statement timeout
#   format_rows()  ──► columns + rows ──► plain-text table the model can read
#
# Does NOT replace a read-only database role. The role is the real
# security boundary; these settings are a second layer.
# ─────────────────────────────────────────────────────────────
MAX_ROWS = 50


def pg_connect(timeout_ms: int = 15000):
    """Open a PostgreSQL connection whose transactions are read-only."""
    kwargs = {
        "host": os.environ["PGHOST"],
        "port": PGPORT,
        "dbname": os.environ["PGDATABASE"],
        "user": os.environ["PGUSER"],
        "password": os.environ["PGPASSWORD"],
        "connect_timeout": 10,
        "row_factory": dict_row,
        "options": f"-c default_transaction_read_only=on -c statement_timeout={timeout_ms}",
    }
    if PGSSLMODE:
        kwargs["sslmode"] = PGSSLMODE
    return psycopg.connect(**kwargs)


def format_rows(columns: list[str], rows: list[dict], truncated: bool) -> str:
    """Render rows as 'col | col' lines. States clearly when rows were cut off."""
    if not rows:
        return "Query succeeded. 0 rows returned."
    header = " | ".join(columns)
    body = "\n".join(" | ".join(str(row.get(c)) for c in columns) for row in rows)
    tail = (f"({len(rows)} rows shown. More rows exist; the result was capped at {MAX_ROWS}.)"
            if truncated else f"({len(rows)} rows.)")
    return f"{header}\n{body}\n{tail}"

In [ ]:
# ─────────────────────────────────────────────────────────────
# run_query(sql) — the agent's HANDS
#
#   sql ──► STEP 1 first-word gate (SELECT / WITH only)
#       ──► STEP 2 execute on a read-only connection
#       ──► STEP 3 format up to 50 rows as text ──► back to the model
#
# Errors are returned as text ("SQL error: ...") so the model can read
# them and fix its query. The tool never raises into the agent loop.
#
# Does NOT check for multiple statements or hidden writes. That is a
# known gap in this version; Version 4 adds a proper SQL policy.
# ─────────────────────────────────────────────────────────────
@tool
def run_query(sql: str) -> str:
    """Execute one read-only SELECT or WITH query against PostgreSQL.

    Returns at most 50 rows as text, or a message starting with
    "SQL error:" or "REFUSED:" that explains what went wrong.
    """
    # STEP 1 — first-word gate.
    sql_clean = sql.strip().rstrip(";").strip()
    if not sql_clean:
        return "REFUSED: SQL query is empty."
    first_word = sql_clean.split(None, 1)[0].upper()
    if first_word not in {"SELECT", "WITH"}:
        return "REFUSED: Only read-only SELECT or WITH queries are allowed."

    # STEP 2 — execute. Fetch one extra row so we know if the result was cut off.
    try:
        with pg_connect() as conn:
            cursor = conn.execute(sql_clean)
            fetched = cursor.fetchmany(MAX_ROWS + 1)
            columns = [d.name for d in cursor.description] if cursor.description else []
    except psycopg.Error as exc:
        return f"SQL error: {exc}"

    # STEP 3 — format for the model.
    return format_rows(columns, fetched[:MAX_ROWS], truncated=len(fetched) > MAX_ROWS)

## 4. The system prompt

Same anatomy as Version 1. We add an **AVAILABLE TOOL** section and rules for using it.

In [ ]:
SYSTEM_PROMPT = """
ROLE
You are an AI Data Analyst Agent for an e-commerce marketplace.

CONTEXT
Known table names include:
orders, order_items, products, customers, shipments, payment_transactions,
reviews, suppliers, warehouses, inventory, promotions, categories,
customer_addresses, product_suppliers
Use only schema details actually known to you. Table names alone do not establish
column names or relationships.

AVAILABLE TOOL
- run_query(sql): executes one read-only query against PostgreSQL and returns up to
  50 rows. It is the only tool available in this version.

TASK AND TOOL USE
- Answer questions about internal marketplace data using PostgreSQL evidence.
- Determine the required metric, filters, aggregation, joins, and date boundaries.
- If needed schema details are missing, do not guess; explain the missing information.
- Construct a single read-only SELECT/WITH query and call run_query(sql).
- Inspect the returned tool result before answering.
- If SQL fails, use the error to correct and retry when possible.
- Never claim a query succeeded unless run_query confirms success.

CONSTRAINTS
- Do not invent table/column names, relationships, business definitions, or results.
- Consider table grain, join cardinality, NULLs, and double-counting.
- For percentages, state the numerator and denominator.
- If the database cannot answer reliably, state UNKNOWN and explain why.
- Do not expose private chain-of-thought; provide a concise explanation only.
- Tool instructions are not a security boundary; database permissions must enforce
  read-only access.

OUTPUT FORMAT
Use relevant sections:
ANSWER: Answer with units where applicable, or UNKNOWN.
DENOMINATOR: For percentages, identify numerator and denominator; otherwise N/A.
SOURCE: Identify PostgreSQL and include the SQL actually executed, if successful.
CAVEAT: State material limitations, ambiguity, or data-quality concerns.
"""

## 5. Create the agent with its tool

Version 1 passed `tools=[]`. Now we pass `[run_query]`.

In [ ]:
agent = create_agent(
    model=model,
    tools=[run_query],
    system_prompt=SYSTEM_PROMPT,
)

## 6. Ask a question and watch the loop

`ask_agent` prints every tool call (🧠 BRAIN → TOOL) and every tool result (🖐️ TOOL → OBSERVATION). We reuse this helper in every later version.

In [ ]:
# ─────────────────────────────────────────────────────────────
# ask_agent(question) — run the agent once and print what happened
#
#   question ──► agent.invoke ──► final state
#                                   │
#                                   ├─ AIMessage with tool_calls ──► "BRAIN → TOOL"
#                                   ├─ ToolMessage               ──► "TOOL → OBSERVATION"
#                                   └─ last message              ──► "FINAL ANSWER"
#
# Does NOT change the agent. It only reads the messages it returns.
# ─────────────────────────────────────────────────────────────
def ask_agent(question: str, agent_to_use=None, max_obs_chars: int = 1500):
    agent_to_use = agent_to_use or agent
    result = agent_to_use.invoke({"messages": [{"role": "user", "content": question}]})

    line = "=" * 80
    print(f"{line}\nQUESTION\n{line}\n{question}")
    print(f"\n{line}\nAGENT TRACE\n{line}")

    for message in result["messages"]:
        kind = type(message).__name__
        if kind == "AIMessage" and getattr(message, "tool_calls", None):
            for call in message.tool_calls:
                print("\n🧠 BRAIN → TOOL")
                print("Tool:", call["name"])
                print("Arguments:", call["args"])
        elif kind == "ToolMessage":
            print(f"\n🖐️ TOOL → OBSERVATION ({message.name})")
            print(str(message.content)[:max_obs_chars])

    print(f"\n{line}\nFINAL ANSWER\n{line}")
    print(result["messages"][-1].content)
    if "sql_call_count" in result:
        print("\nrun_query executions used:", result["sql_call_count"])
    return result

In [ ]:
result = ask_agent("How many orders were cancelled?")

## 7. More questions

The agent only knows table names, so watch for SQL errors on unknown columns and how it retries. That weakness is what Version 3 fixes.

In [ ]:
for question in [
    "What are the top 10 best-selling products?",
    "What percentage of orders were cancelled?",
    "Which products have inventory below 100 units?",
]:
    ask_agent(question)

## 8. Key points

```text
question ──► brain decides it needs data ──► run_query(sql) ──► PostgreSQL
                                                    │
answer   ◄── brain reads the observation ◄──────────┘
```

- The tool runs SQL. The model only reads the result.
- Errors come back as text, so the model can correct itself.
- The agent still guesses column names. That causes avoidable errors.

**Limitations:** use a read-only role; the 50-row cap limits rows returned, not database work.

**Next — Version 3:** the agent looks up schema, examples and business definitions in Pinecone before writing SQL.